In [ ]:
from google.colab import drive

drive.mount("/content/drive")

# 01 — Baseline Performance Comparison

Membandingkan LDA, NMF, ProdLDA, NVDM, CombinedTM, BERTopic, Top2Vec,
serta empat variasi HMRNet dengan metrik topik yang seragam.


In [ ]:
from pathlib import Path
import sys

# Sesuaikan hanya apabila folder proyek berada di lokasi lain.
PROJECT_ROOT = Path("/content/drive/MyDrive/UDB/Riset/2026")
if not PROJECT_ROOT.exists():
    DRIVE_ROOT = Path("/content/drive/MyDrive/HMRNet_Experiments")
    if DRIVE_ROOT.exists():
        PROJECT_ROOT = DRIVE_ROOT

assert PROJECT_ROOT.exists(), (
    "Folder HMRNet_Experiments tidak ditemukan. "
    "Unggah/ekstrak proyek ke /content atau Google Drive."
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
TABLES_DIR = RESULTS_DIR / "tables"
FIGURES_DIR = RESULTS_DIR / "figures"
CHECKPOINTS_DIR = RESULTS_DIR / "checkpoints"
LOGS_DIR = RESULTS_DIR / "logs"
FEATURE_DIR = CHECKPOINTS_DIR / "features"

for directory in [TABLES_DIR, FIGURES_DIR, CHECKPOINTS_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
import numpy as np
import pandas as pd
from src.preprocessing import load_feature_bundle

bundle = load_feature_bundle(FEATURE_DIR)
raw_texts = bundle["raw_texts"]
processed_texts = bundle["processed_texts"]
bow = bundle["bow"]
bow_normalized = bundle["bow_normalized"]
tfidf = bundle["tfidf"]
semantic_embeddings = bundle["semantic_embeddings"]
document_embeddings = bundle["document_embeddings"]
vocabulary = bundle["vocabulary"]

print("Documents:", len(raw_texts))
print("Vocabulary:", len(vocabulary))
print("Semantic shape:", semantic_embeddings.shape)
print("Document shape:", document_embeddings.shape)


In [ ]:
import joblib
import pandas as pd
import torch

from src.baseline_models import (
    fit_bertopic,
    fit_combined_tm,
    fit_lda,
    fit_nmf,
    fit_nvdm,
    fit_prodlda,
    fit_top2vec,
)
from src.hmrnet import HMRNet, HMRNetConfig
from src.topic_metrics import evaluate_many_models

N_TOPICS = 20
TOP_N_WORDS = 10
QUICK_MODE = False
RUN_HEAVY_BASELINES = True
EPOCHS = 15 if QUICK_MODE else 60


In [ ]:
model_results = {}
failures = {}

def register(result):
    model_results[result.name] = result
    print(f"✓ {result.name}: {len(result.topics)} topics")


In [ ]:
register(
    fit_lda(
        bow=bow,
        vocabulary=vocabulary,
        n_topics=N_TOPICS,
        top_n=TOP_N_WORDS,
        random_state=42,
        max_iter=10 if QUICK_MODE else 30,
    )
)

register(
    fit_nmf(
        tfidf=tfidf,
        vocabulary=vocabulary,
        n_topics=N_TOPICS,
        top_n=TOP_N_WORDS,
        random_state=42,
    )
)


In [ ]:
register(
    fit_prodlda(
        bow=bow,
        vocabulary=vocabulary,
        n_topics=N_TOPICS,
        top_n=TOP_N_WORDS,
        epochs=EPOCHS,
        random_state=42,
    )
)

register(
    fit_nvdm(
        bow=bow,
        vocabulary=vocabulary,
        n_topics=N_TOPICS,
        top_n=TOP_N_WORDS,
        epochs=EPOCHS,
        random_state=42,
    )
)


In [ ]:
%pip install -q --upgrade pip setuptools wheel

%pip install -q \
    contextualized-topic-models==2.6.1 \
    bertopic==0.17.4 \
    top2vec==1.0.36

Periksa hasil instalasi

In [ ]:
import sys
import importlib.metadata as metadata

print("Python:", sys.version)

packages = {
    "contextualized-topic-models": "contextualized-topic-models",
    "BERTopic": "bertopic",
    "Top2Vec": "top2vec",
}

for name, package in packages.items():
    try:
        version = metadata.version(package)
        print(f"✓ {name}: {version}")
    except metadata.PackageNotFoundError:
        print(f"✗ {name}: belum terpasang")

Uji impor modul

In [ ]:
try:
    from contextualized_topic_models.models.ctm import CombinedTM
    print("✓ CombinedTM berhasil diimpor")
except Exception as error:
    print("✗ CombinedTM:", repr(error))

try:
    from bertopic import BERTopic
    print("✓ BERTopic berhasil diimpor")
except Exception as error:
    print("✗ BERTopic:", repr(error))

try:
    from top2vec import Top2Vec
    print("✓ Top2Vec berhasil diimpor")
except Exception as error:
    print("✗ Top2Vec:", repr(error))

Periksa dependensi yang sering dibutuhkan

In [ ]:
import numpy
import pandas
import sklearn
import torch
import gensim
import sentence_transformers
import umap
import hdbscan

print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("Scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("Gensim:", gensim.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("UMAP:", umap.__version__)
print("HDBSCAN:", hdbscan.__version__ if hasattr(hdbscan, "__version__") else "installed")

In [ ]:
RUN_HEAVY_BASELINES = True

In [ ]:
if RUN_HEAVY_BASELINES:
    heavy_functions = {
        "CombinedTM": lambda: fit_combined_tm(
            raw_texts=raw_texts,
            processed_texts=processed_texts,
            n_topics=N_TOPICS,
            top_n=TOP_N_WORDS,
            epochs=10 if QUICK_MODE else 50,
            random_state=42,
        ),
        "BERTopic": lambda: fit_bertopic(
            raw_texts=raw_texts,
            n_topics=N_TOPICS,
            top_n=TOP_N_WORDS,
            random_state=42,
        ),
        "Top2Vec": lambda: fit_top2vec(
            raw_texts=raw_texts,
            top_n=TOP_N_WORDS,
            embedding_model="doc2vec",
            speed="fast-learn" if QUICK_MODE else "learn",
        ),
    }

    for name, function in heavy_functions.items():
        try:
            register(function())
        except Exception as error:
            failures[name] = repr(error)
            print(f"✗ {name}: {error}")


In [ ]:
print(model_results.keys())

In [ ]:
variants = {
    "HMRNet-Lexical": dict(
        use_lexical=True,
        use_semantic=False,
        use_document=False,
        use_cra=False,
        fusion_mode="average",
    ),
    "HMRNet-Semantic": dict(
        use_lexical=False,
        use_semantic=True,
        use_document=False,
        use_cra=False,
        fusion_mode="average",
    ),
    "HMRNet-Document": dict(
        use_lexical=False,
        use_semantic=False,
        use_document=True,
        use_cra=False,
        fusion_mode="average",
    ),
    "HMRNet-Full": dict(
        use_lexical=True,
        use_semantic=True,
        use_document=True,
        use_cra=True,
        fusion_mode="adaptive",
    ),
}

for name, overrides in variants.items():
    config = HMRNetConfig(
        n_topics=N_TOPICS,
        epochs=EPOCHS,
        patience=5 if QUICK_MODE else 8,
        random_state=42,
        **overrides,
    )
    model = HMRNet(config).fit(
        bow=bow,
        semantic_embeddings=semantic_embeddings,
        document_embeddings=document_embeddings,
        vocabulary=vocabulary,
    )
    result = type("Result", (), {})()
    result.name = name
    result.topics = model.get_topics(TOP_N_WORDS)
    result.topic_word = model.get_topic_word_distribution()
    result.document_topic = model.transform(
        bow, semantic_embeddings, document_embeddings
    )
    result.model = model
    result.metadata = {}
    model_results[name] = result

    model.save(CHECKPOINTS_DIR / f"{name}.pt")
    print(f"✓ {name}")


In [ ]:
topic_dictionary = {
    name: result.topics
    for name, result in model_results.items()
}
comparison = evaluate_many_models(
    topic_dictionary,
    processed_texts=processed_texts,
    top_n=TOP_N_WORDS,
)

comparison.to_csv(
    TABLES_DIR / "experiment_01_baseline_comparison.csv",
    index=False,
)
comparison.to_excel(
    TABLES_DIR / "experiment_01_baseline_comparison.xlsx",
    index=False,
)
display(comparison.round(4))

if failures:
    print("Baseline yang dilewati:", failures)


In [ ]:
# Simpan objek baseline yang dapat diserialisasi.
serializable = {
    name: result
    for name, result in model_results.items()
    if name in {"LDA", "NMF"}
}
joblib.dump(
    serializable,
    CHECKPOINTS_DIR / "baseline_sklearn_models.joblib",
)

# Simpan representasi dan kata topik seluruh model secara portabel.
portable = {}
for name, result in model_results.items():
    portable[name] = {
        "topics": result.topics,
        "topic_word": result.topic_word,
        "document_topic": result.document_topic,
    }
joblib.dump(portable, CHECKPOINTS_DIR / "experiment_01_outputs.joblib")


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/UDB/Riset/2026")
TABLES_DIR = PROJECT_ROOT / "results/tables"
CHECKPOINTS_DIR = PROJECT_ROOT / "results/checkpoints"

required_outputs = [
    TABLES_DIR / "experiment_01_baseline_comparison.csv",
    TABLES_DIR / "experiment_01_baseline_comparison.xlsx",
    CHECKPOINTS_DIR / "experiment_01_outputs.joblib",
    CHECKPOINTS_DIR / "HMRNet-Full.pt",
]

for file_path in required_outputs:
    print(f"{file_path.name}: {file_path.exists()}")